# mini-vLLM — GPU checks (Colab T4)

Runs the GPU-only checks in order. Nothing here is a benchmark claim until you have looked at the output.

- **a** fp16 engine vs HF `generate` in the same dtype: top-1 agreement (target over 99%) and first divergence
- **b** memory pressure: a small KV pool forces preemption; every request must still finish
- **c** sizing: the P24-predicted block count allocates without OOM; measured peak vs prediction
- **d** the identical-outputs gate (sequential, static, engine) on GPU
- **e** the sweep, then plots

**Before you run:** Colab clones `origin/master`. Commit and push the working tree first, or the clone won't contain these changes.
P26 (vLLM comparison, cost-model fit) waits until you have looked at the output of **e**.

In [ ]:
!git clone https://github.com/AustinD123/mini-vllm.git
%cd mini-vllm
!pip install -q transformers matplotlib pytest
import sys
sys.path.insert(0, ".")

## Load the model twice: HF attention (reference) and the paged backend (engine)

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from minivllm.attention.backend import BACKEND_NAME, register_backend

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DTYPE = torch.float16
DEVICE = "cuda"
assert torch.cuda.is_available(), "switch the runtime to a GPU (Runtime > Change runtime type)"

register_backend()
ref = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, attn_implementation="sdpa").eval().to(DEVICE)
paged = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, attn_implementation=BACKEND_NAME).eval().to(DEVICE)
cfg = ref.config
print(MODEL_ID, DTYPE, torch.cuda.get_device_name(0), "torch", torch.__version__)

## Seeded workload (same generator as the sweep, so every check sees the same requests)

In [ ]:
from minivllm.bench.workload import make_workload
from minivllm.bench.jobs import build_jobs

SEED = 0
requests = make_workload(rate=10.0, num_requests=16, seed=SEED, prompt_mu=4.0, output_mu=3.5, max_len=128)
jobs = build_jobs(requests, vocab_size=cfg.vocab_size, seed=SEED)
prompts = [list(j.prompt_ids) for j in jobs]
lens = [j.output_len for j in jobs]
print(len(jobs), "requests; prompt lengths", [len(p) for p in prompts][:8], "...")

## a. fp16 engine vs HF generate, same dtype. Headline: free-running agreement and first divergence. Teacher-forced is the per-step check.

In [ ]:
from minivllm.bench.gpu_checks import agreement, run_engine, teacher_forced_agreement

ref_out = {}
for j in jobs:
    ids = torch.tensor([list(j.prompt_ids)], device=DEVICE)
    with torch.no_grad():
        out = ref.generate(ids, do_sample=False, max_new_tokens=j.output_len, min_new_tokens=j.output_len,
                           eos_token_id=None, pad_token_id=0, repetition_penalty=1.0)
    ref_out[j.request_id] = out[0, len(j.prompt_ids):].tolist()

engine_out, preemptions = run_engine(paged, prompts, lens, num_blocks=512, max_batch=8, token_budget=512)
a = agreement(engine_out, ref_out)
print(f"top-1 agreement: {a['top1']:.4%} over {a['positions']} positions (target > 99%)")
print("first divergence per request:", a["first_divergence"])
print("preemptions:", preemptions)
t = teacher_forced_agreement(paged, prompts, ref_out)
print(f"teacher-forced top-1 agreement: {t['top1']:.4%} over {t['positions']} positions (target > 99%)")

## b. memory pressure: a shrunken pool must preempt, still finish, and still agree

In [ ]:
import math
need = [math.ceil((len(p) + n) / 16) for p, n in zip(prompts, lens)]
small_pool = max(need) + 2  # enough for the largest request alone, not for the whole batch

pressed_out, pressed_preemptions = run_engine(paged, prompts, lens, num_blocks=small_pool, max_batch=8, token_budget=512)
finished = all(len(pressed_out[j.request_id]) == j.output_len for j in jobs)
b = agreement(pressed_out, ref_out)
print("pool blocks:", small_pool, "| preemptions:", pressed_preemptions)
print("all requests finished:", finished)
print(f"top-1 agreement under pressure: {b['top1']:.4%}")
assert finished, "some request did not finish under memory pressure"

## c. sizing: allocate the P24-predicted blocks; compare measured peak with the prediction

In [ ]:
from minivllm.bench.gpu_checks import sizing_check

KV_FRACTION = 0.5
sizing = sizing_check(cfg, DTYPE, fraction=KV_FRACTION)
for key, value in sizing.items():
    print(f"{key:>22}: {value}")

## d. the identical-outputs gate on GPU (fp64 tiny model; same test as CPU)

In [ ]:
!python -m pytest -q tests/test_baselines.py -k gate

## e. sweep, then plots

In [ ]:
from minivllm.bench.plots import plot_sweep, sweep_table
from minivllm.bench.sweep import run_sweep
from minivllm.bench.workload import LENGTH_DEFAULTS
from minivllm.clock import WallClock
from minivllm.bench.results import validate_result
from IPython.display import Image, display

records = run_sweep(
    ref, paged,
    rates=[1.0, 2.0, 4.0, 8.0],
    num_requests=64,
    seed=SEED,
    out_dir="results/gpu",
    clock_factory=WallClock,
    vocab_size=cfg.vocab_size,
    max_batch=16,
    token_budget=512,
    kv_memory_fraction=KV_FRACTION,  # num_blocks=None: the pool is sized from this share of free GPU memory
    length_overrides=dict(LENGTH_DEFAULTS, max_len=512),
)
for r in records:
    validate_result(r)
for row in sweep_table(records):
    print(row)

In [ ]:
paths = plot_sweep(records, "results/gpu/plots")
for p in paths:
    display(Image(str(p)))